# 👍 Lab 7.2 — Collect & Store User Feedback via API

## Learning Objectives
In this notebook, you will learn:
1. **`client_request_id` Contract** — Tag every trace with the UI-supplied request ID — the join key for your `/api/feedback` endpoint
2. **Trace Lookup** — Resolve `client_request_id` → `trace_id` with `mlflow.search_traces(filter_string=…)`
3. **`mlflow.log_feedback`** — Attach typed `Feedback` assessments with `AssessmentSource(source_type="HUMAN", source_id=…)`
4. **Multi-Field Feedback** — Helpful + free-text rationale + numeric satisfaction — multiple assessments per trace
5. **Human vs. Machine View** — Compare aggregate human feedback against `Correctness`/`Safety` judge scores on the same cohort
6. **Production-Shaped Endpoint** — Wrap the lookup + log_feedback into a `submit_user_feedback(...)` function

## Prerequisites
- Lab 1.3 (workspace + experiment)
- Lab 4.2 (built-in judges)
- Lab 6.5 (registered scorers — useful for the comparison view)


In [ ]:
# ============================================================================
# 📦 INSTALL PACKAGES
# ============================================================================

%pip install --quiet "mlflow[databricks]>=3.1"

dbutils.library.restartPython()


---
## Step 1 — Configure Experiment


In [ ]:
# ============================================================================
# 🧪 STEP 1 - CONFIGURE EXPERIMENT
# ============================================================================

import mlflow

USER_EMAIL = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook()
    .getContext()
    .userName()
    .get()
)
EXPERIMENT_PATH = f"/Users/{USER_EMAIL}/genai-eval-tutorial"
mlflow.set_experiment(EXPERIMENT_PATH)

print(f"Experiment: {EXPERIMENT_PATH}")


---
## Step 2 — The Feedback Loop in One Picture

```
 ┌───────────────┐    client_request_id     ┌──────────────┐
 │  Your UI      │ ─────────────────────▶  │   Agent      │
 │ (ChatBox)     │                         │  @mlflow.trace│
 └─────┬─────────┘                         └──────┬───────┘
       │  thumbs-up                                │ logs trace
       │  client_request_id="abc"                  ▼
       │                                  ┌──────────────────┐
       ▼                                  │ MLflow trace store│
 /api/feedback ───── search_traces ─────▶  └──────────────────┘
       │                                          │
       └──── log_feedback(trace_id, …) ──────────▶│
```

The contract is one piece of data: **`client_request_id`**. Anything your UI can persist next to a feedback button is a valid choice — request UUID, conversation turn ID, etc.


---
## Step 3 — Build a Tiny Agent That Tags Each Trace with `client_request_id`

The pattern: take `client_request_id` as an argument, set it as a **trace tag** in the very first line of the function. That makes it queryable from `search_traces` later.


In [ ]:
# ============================================================================
# 🔭 STEP 3 - BUILD A TINY AGENT THAT TAGS EACH TRACE WITH `CLIENT_REQUEST_ID`
# ============================================================================

import uuid
from databricks.sdk import WorkspaceClient

oai = WorkspaceClient().serving_endpoints.get_open_ai_client()

@mlflow.trace
def chat_agent(question: str, client_request_id: str) -> str:
    # Tag THIS trace with the UI-provided ID so the feedback endpoint can find it later.
    mlflow.update_current_trace(tags={"client_request_id": client_request_id})

    resp = oai.chat.completions.create(
        model="databricks-claude-opus-4-6",
        messages=[
            {"role": "system", "content": "You are a Databricks expert. Answer concisely."},
            {"role": "user",   "content": question},
        ],
    )
    return resp.choices[0].message.content


---
## Step 4 — Send 5 Traced Requests with Unique `client_request_id`s


In [ ]:
# ============================================================================
# 🔭 STEP 4 - SEND 5 TRACED REQUESTS WITH UNIQUE `CLIENT_REQUEST_ID`S
# ============================================================================

QUESTIONS = [
    "What is Delta Lake?",
    "Explain Z-ordering.",
    "How does Unity Catalog handle column-level lineage?",
    "What is the default VACUUM retention?",
    "Compare partitioning and Z-ordering.",
]

requests_log = []  # what your UI would persist alongside the chat history
for q in QUESTIONS:
    crid = f"req-{uuid.uuid4().hex[:8]}"
    answer = chat_agent(q, client_request_id=crid)
    requests_log.append({"client_request_id": crid, "question": q, "answer": answer})

for r in requests_log:
    print(f"{r['client_request_id']}  ▸  {r['question']}")


---
## Step 5 — Resolve `client_request_id` → `trace_id`

This is the lookup your `/api/feedback` endpoint runs on each user click. It's exactly **one** `search_traces` call.


In [ ]:
# ============================================================================
# 🔭 STEP 5 - RESOLVE `CLIENT_REQUEST_ID` → `TRACE_ID`
# ============================================================================

def trace_id_for(client_request_id: str) -> str:
    df = mlflow.search_traces(
        experiment_names=[EXPERIMENT_PATH],
        filter_string=f"tags.client_request_id = '{client_request_id}'",
        max_results=1,
    )
    rows = df.collect() if hasattr(df, "collect") else df.to_dict("records")
    if not rows:
        raise LookupError(f"No trace found for client_request_id={client_request_id}")
    row = rows[0]
    return row["trace_id"] if isinstance(row, dict) else row.trace_id

# Sanity check on the first request
sample_crid = requests_log[0]["client_request_id"]
sample_trace = trace_id_for(sample_crid)
print(f"client_request_id={sample_crid} → trace_id={sample_trace}")


---
## Step 6 — Simulate a User Thumbs-Up: `mlflow.log_feedback(...)`

`log_feedback` attaches a typed **`Feedback`** assessment to the trace. Three things matter:

| Field | Why |
| --- | --- |
| `name`      | column name in eval results — keep it stable across product versions |
| `value`     | the rating itself: bool, str, or numeric |
| `source`    | `AssessmentSource(source_type="HUMAN", source_id="user_123")` — provenance for filtering & calibration |


In [ ]:
# ============================================================================
# 🔭 STEP 6 - SIMULATE A USER THUMBS-UP: `MLFLOW.LOG_FEEDBACK(...)`
# ============================================================================

from mlflow.entities import AssessmentSource

# Simulate two thumbs-ups, two thumbs-downs, one neutral
SIMULATED_FEEDBACK = [
    {"crid": requests_log[0]["client_request_id"], "user": "user_123", "value": True,  "rationale": "Concise, accurate."},
    {"crid": requests_log[1]["client_request_id"], "user": "user_456", "value": True,  "rationale": "Z-order explanation matched my mental model."},
    {"crid": requests_log[2]["client_request_id"], "user": "user_789", "value": False, "rationale": "Missed the column-level lineage detail."},
    {"crid": requests_log[3]["client_request_id"], "user": "user_456", "value": False, "rationale": "Said retention is 30 days, actual default is 7."},
    {"crid": requests_log[4]["client_request_id"], "user": "user_321", "value": True,  "rationale": "Good comparison table."},
]

for fb in SIMULATED_FEEDBACK:
    tid = trace_id_for(fb["crid"])
    mlflow.log_feedback(
        trace_id=tid,
        name="user_helpful",
        value=fb["value"],
        rationale=fb["rationale"],
        source=AssessmentSource(source_type="HUMAN", source_id=fb["user"]),
    )

print("✅ 5 user feedback rows logged.")


---
## Step 7 — Log a Numeric Satisfaction Score Alongside

Most product UIs collect more than thumbs — a 1-5 star rating, a topic tag, a free-text comment. Each one is a separate `log_feedback` call with its own `name`. They all attach to the same trace.


In [ ]:
# ============================================================================
# ▶️ STEP 7 - LOG A NUMERIC SATISFACTION SCORE ALONGSIDE
# ============================================================================

SATISFACTION = {
    requests_log[0]["client_request_id"]: 5,
    requests_log[1]["client_request_id"]: 4,
    requests_log[2]["client_request_id"]: 2,
    requests_log[3]["client_request_id"]: 1,
    requests_log[4]["client_request_id"]: 5,
}

for crid, score in SATISFACTION.items():
    mlflow.log_feedback(
        trace_id=trace_id_for(crid),
        name="satisfaction_1_5",
        value=score,
        source=AssessmentSource(source_type="HUMAN", source_id="aggregate"),
    )

print("✅ Satisfaction scores logged.")


---
## Step 8 — Pull All Feedback for the Test Cohort

Each trace's `assessments` column contains every assessment ever attached — humans + automated judges + registered scorers. Filter by `source.source_type` to slice human vs. machine.


In [ ]:
# ============================================================================
# ▶️ STEP 8 - PULL ALL FEEDBACK FOR THE TEST COHORT
# ============================================================================

crids = [r["client_request_id"] for r in requests_log]
crid_filter = " OR ".join(f"tags.client_request_id = '{c}'" for c in crids)

cohort = mlflow.search_traces(
    experiment_names=[EXPERIMENT_PATH],
    filter_string=crid_filter,
    max_results=20,
)
display(cohort)


---
## Step 9 — Compare Human Feedback vs. Automated Judges

Run an automated judge (`Correctness`) over the same questions, then put both score columns side-by-side. The point isn't "humans are right and judges are wrong" — it's that you can now **measure agreement** (Lab 7.3 turns this into a calibration loop).


In [ ]:
# ============================================================================
# ▶️ STEP 9 - COMPARE HUMAN FEEDBACK VS. AUTOMATED JUDGES
# ============================================================================

import pandas as pd
from mlflow.genai.scorers import Correctness, Safety

# Build a small eval dataframe matching the 5 questions, with empty expected_facts.
# Correctness will grade against `expected_response` if provided; without it, it
# uses LLM-as-judge to verify the answer is plausible. We pre-fill expected_facts
# from the inference table semantics for clarity.

EXPECTED_FACTS = {
    "What is Delta Lake?":                                 ["ACID transactions", "schema enforcement", "time travel"],
    "Explain Z-ordering.":                                 ["co-locates related data", "improves data skipping"],
    "How does Unity Catalog handle column-level lineage?": ["column-level lineage", "Unity Catalog"],
    "What is the default VACUUM retention?":               ["7 days"],
    "Compare partitioning and Z-ordering.":                ["partitioning", "Z-ordering"],
}

eval_df = pd.DataFrame([
    {"inputs": {"question": q}, "expectations": {"expected_facts": EXPECTED_FACTS[q]}}
    for q in QUESTIONS
])

@mlflow.trace
def predict_for_eval(question: str) -> str:
    return chat_agent(question, client_request_id=f"eval-{uuid.uuid4().hex[:8]}")

results = mlflow.genai.evaluate(
    data=eval_df,
    predict_fn=predict_for_eval,
    scorers=[Correctness(), Safety()],
    model_id="models:/feedback-cohort/v1",
)

display(results.tables["eval_results"].select(
    "inputs",
    "outputs",
    "`correctness/v1/value`",
    "`safety/v1/value`",
))


In [ ]:
# ============================================================================
# ▶️ SIDE-BY-SIDE: HUMAN FEEDBACK VS. AUTOMATED JUDGES
# ============================================================================

import pandas as pd

human_df = pd.DataFrame([
    {"question": next(r["question"] for r in requests_log if r["client_request_id"] == fb["crid"]),
     "user_helpful":  fb["value"],
     "satisfaction":  SATISFACTION[fb["crid"]]}
    for fb in SIMULATED_FEEDBACK
])

print("Human feedback aggregate:")
print(f"  thumbs_up_rate  = {human_df['user_helpful'].mean():.2f}")
print(f"  mean_satisfaction = {human_df['satisfaction'].mean():.2f}/5")

judge_df = (results.tables["eval_results"]
                .selectExpr(
                    "AVG(CASE WHEN `correctness/v1/value` = 'yes' THEN 1.0 ELSE 0.0 END) AS correctness_pass",
                    "AVG(CASE WHEN `safety/v1/value`      = 'yes' THEN 1.0 ELSE 0.0 END) AS safety_pass",
                ).toPandas())
print("\nAutomated judge aggregate:")
print(judge_df.to_string(index=False))


---
## Step 10 — What Your `/api/feedback` Endpoint Looks Like

Wrap Steps 5 + 6 into a function and you have the production endpoint. Hosted as a Databricks App, FastAPI service, or serverless function — same code.


In [ ]:
# ============================================================================
# ▶️ STEP 10 - WHAT YOUR `/API/FEEDBACK` ENDPOINT LOOKS LIKE
# ============================================================================

def submit_user_feedback(client_request_id: str,
                         user_id: str,
                         helpful: bool,
                         rationale: str | None = None,
                         satisfaction: int | None = None) -> dict:
    """Production-shaped feedback handler — call from your UI's API."""
    tid = trace_id_for(client_request_id)
    mlflow.log_feedback(
        trace_id=tid,
        name="user_helpful",
        value=helpful,
        rationale=rationale,
        source=AssessmentSource(source_type="HUMAN", source_id=user_id),
    )
    if satisfaction is not None:
        mlflow.log_feedback(
            trace_id=tid,
            name="satisfaction_1_5",
            value=satisfaction,
            source=AssessmentSource(source_type="HUMAN", source_id=user_id),
        )
    return {"trace_id": tid, "ok": True}

# Example UI call:
print(submit_user_feedback(
    client_request_id=requests_log[0]["client_request_id"],
    user_id="user_999",
    helpful=True,
    rationale="Used this answer in my onboarding doc.",
    satisfaction=5,
))


---
## Lab Complete

| Check | Status |
| --- | --- |
| 5 requests sent, each tagged with a unique `client_request_id` | ✅ |
| `search_traces(filter_string=…)` resolves `client_request_id` → `trace_id` | ✅ |
| `log_feedback` attached typed human assessments (helpful + satisfaction) | ✅ |
| Same traces also have automated judge scores from `Correctness` / `Safety` | ✅ |
| Side-by-side aggregate of human vs. machine scores produced | ✅ |
| Production-shaped `submit_user_feedback()` endpoint demonstrated | ✅ |

Next: **Lab 7.3** — turn that human feedback into a **judge calibration loop** — measure agreement, refine the rubric, re-evaluate.

After that: **Lab 7.4** — wire the calibrated judges into a **pre-deployment quality gate** in Databricks Workflows.


---
## 📝 Summary

In this notebook, we covered:

### 1. Why the Loop Matters
- Human feedback is the ground truth that calibrates everything else (Lab 7.3).
- The contract is small — `client_request_id` flows from UI → trace tag → feedback endpoint.
- Once the loop is closed, every product event becomes labelled training data for free.

### 2. Three Things to Get Right
- **Stable `name`** — column name in eval results; don't rename across product versions.
- **Correct `source`** — `AssessmentSource(source_type="HUMAN", source_id=user_id)`.
- **One assessment per fact** — separate `log_feedback` calls for helpful, satisfaction, comment.

### 3. From Raw Feedback to Calibration
- Aggregate human feedback alongside automated judge scores in MLflow.
- Disagreement between humans and judges is the signal Lab 7.3 uses to refine rubrics.

### Next Steps
- **Lab 7.3** — judge calibration loop: measure agreement, refine the rubric, re-evaluate.
